In [43]:
import inspect
import fefetlab.b1500.driver as drv

print(inspect.signature(drv.B1500.dv))

(self, ch: 'int', vrange: 'int', voltage: 'float', compliance: 'float | None' = None, comp_polarity: 'int | None' = None, irange: 'int | None' = None) -> 'None'


In [44]:
 # 单独测试通道4
from fefetlab.instruments.visa_session import VisaSession, VisaConfig
from fefetlab.b1500 import B1500

cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=60000,          # 先拉长，避免 10 s 太紧
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

with VisaSession(cfg) as s:
    b = B1500(s)

    print("IDN:", s.query("*IDN?"))
    print("ERRX start:", b.clear_err_queue())

    # 更接近手册的 high-speed spot 配置
    s.write("FMT 1")
    s.write("AV 10,1")
    s.write("FL 0")

    b.cn([4])

    # 你当前 driver 的 dv 签名是 dv(ch, vrange, voltage, compliance, ...)
    b.dv(4, 0, 0.0, 1e-3)

    raw_resp = s.query("TI 4,0")
    print("Raw TI response:", repr(raw_resp))

    print("ERRX end:", b.clear_err_queue())

    b.dz([4])
    b.cl([4])

IDN: Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401
ERRX start: ['+0,"No Error."']
Raw TI response: 'NDI-000.750E-12'
ERRX end: ['+0,"No Error."']


In [45]:
print(b._parse_scalar_response("NDI-010.895E-12"))

-1.0895e-11


In [46]:
print(type(b))
print(dir(b))

<class 'fefetlab.b1500.driver.B1500'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_extract_status', '_format_channels', '_is_no_error', '_normalize_channels', '_parse_errx_code', '_parse_scalar_response', '_query', '_validate_channel', '_write', 'cl', 'clear_err_queue', 'cn', 'dv', 'dz', 'errx', 'fmt', 'opc', 'reset', 's', 'ti']


In [50]:
import time
import pandas as pd

def spot_i_check_one_channel(b, ch, force_v=0.0, icomp=1e-3, delay_s=0.05):
    result = {
        "ch": ch,
        "raw": None,
        "current_A": None,
        "quality": None,
        "error": None,
    }

    try:
        b._write("FMT 1")
        time.sleep(delay_s)

        b._write("AV 10,1")
        time.sleep(delay_s)

        b._write("FL 0")
        time.sleep(delay_s)

        # 这里改成 [ch]
        b.cn([ch])
        time.sleep(delay_s)

        b.dv(ch, 0, force_v, icomp)
        time.sleep(delay_s)

        raw = b._query(f"TI {ch},0", check_err=False)
        result["raw"] = raw

        val = b._parse_scalar_response(raw)
        result["current_A"] = val

        if abs(val) <= 1e-6:
            result["quality"] = "valid"
        else:
            result["quality"] = "suspicious"

    except Exception as e:
        result["quality"] = "invalid"
        result["error"] = repr(e)

    finally:
        try:
            b.dz([ch])
        except Exception:
            pass

        try:
            b.cl([ch])
        except Exception:
            pass

    return result

channels = [4, 5, 6, 7]
rows = []

for ch in channels:
    print(f"\n========== 测试 CH{ch} ==========")
    res = spot_i_check_one_channel(b, ch, force_v=0.0, icomp=1e-3, delay_s=0.05)

    print("raw      =", repr(res["raw"]))
    print("current  =", res["current_A"])
    print("quality  =", res["quality"])
    print("error    =", res["error"])

    rows.append(res)

df = pd.DataFrame(rows)

print("\n=== 汇总结果 ===")
print(df)


========== 测试 CH4 ==========
raw      = 'NDI-005.435E-12\r'
current  = -5.435e-12
quality  = valid
error    = None

========== 测试 CH5 ==========
raw      = 'NEI-008.625E-12\r'
current  = -8.625e-12
quality  = valid
error    = None

========== 测试 CH6 ==========
raw      = 'NFI+005.025E-12\r'
current  = 5.025e-12
quality  = valid
error    = None

========== 测试 CH7 ==========
raw      = 'NGI-0.00040E-09\r'
current  = -4e-13
quality  = valid
error    = None

=== 汇总结果 ===
   ch                raw     current_A quality error
0   4  NDI-005.435E-12\r -5.435000e-12   valid  None
1   5  NEI-008.625E-12\r -8.625000e-12   valid  None
2   6  NFI+005.025E-12\r  5.025000e-12   valid  None
3   7  NGI-0.00040E-09\r -4.000000e-13   valid  None


In [48]:
print("b =", b)
print("has s:", hasattr(b, "s"))
print("b.s =", b.s)
print("type(b.s) =", type(b.s))

try:
    print("IDN =", b.s.query("*IDN?"))
except Exception as e:
    print("query *IDN? 失败：", repr(e))

b = <fefetlab.b1500.driver.B1500 object at 0x000002119EE2ED50>
has s: True
b.s = <fefetlab.instruments.visa_session.VisaSession object at 0x000002119EE2E550>
type(b.s) = <class 'fefetlab.instruments.visa_session.VisaSession'>
query *IDN? 失败： RuntimeError('VISA session is not opened.')


In [49]:
import pyvisa

VISA_ADDR = "GPIB0::17::INSTR"

rm = pyvisa.ResourceManager()
print("VISA resources:", rm.list_resources())

s = rm.open_resource(VISA_ADDR)

# 建议加上终止符和超时
s.timeout = 10000
s.write_termination = "\n"
s.read_termination = "\n"

print("*IDN? ->", s.query("*IDN?"))

# 把新的 session 绑回 b
b.s = s

print("重新绑定后，再试一次：")
print("*IDN? ->", b.s.query("*IDN?"))

VISA resources: ('ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL7::INSTR', 'ASRL8::INSTR', 'ASRL9::INSTR', 'ASRL10::INSTR', 'ASRL11::INSTR', 'GPIB0::17::INSTR')
*IDN? -> Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401
重新绑定后，再试一次：
*IDN? -> Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401


## 测试看看是不是4567某一路不通

In [1]:
import time
from fefetlab.instruments.visa_session import VisaSession, VisaConfig
from fefetlab.b1500 import B1500
cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=30000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

In [3]:
def probe_single_channel(ch: int):
    with VisaSession(cfg) as s:
        b = B1500(s)

        result = {"ch": ch}

        try:
            result["idn"] = s.query("*IDN?")
            result["err_before"] = b.clear_err_queue()

            s.write("FMT 1")
            s.write("AV 10,1")
            s.write("FL 0")

            s.write(f"CN {ch}")
            s.write(f"DV {ch},0,0,1E-3")
            time.sleep(0.2)

            result["raw"] = s.query(f"TI {ch},0")
            result["lop"] = s.query("LOP?")
            result["err_after"] = s.query("ERRX?")

        except Exception as e:
            result["exception"] = repr(e)
            try:
                result["lop_on_error"] = s.query("LOP?")
            except Exception as e2:
                result["lop_on_error"] = repr(e2)
            try:
                result["err_on_error"] = s.query("ERRX?")
            except Exception as e3:
                result["err_on_error"] = repr(e3)

        finally:
            try:
                s.write(f"DZ {ch}")
            except Exception:
                pass
            try:
                s.write(f"CL {ch}")
            except Exception:
                pass

        return result



In [4]:
results = []

for ch in [4, 5, 6, 7]:
    r = probe_single_channel(ch)
    results.append(r)
    print("=" * 60)
    print(r)

{'ch': 4, 'idn': 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401', 'err_before': ['+0,"No Error."'], 'raw': 'NDI+002.135E-12', 'lop': 'LOP00,00,00,01,01,01,00,00,00,00', 'err_after': '+0,"No Error."'}
{'ch': 5, 'idn': 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401', 'err_before': ['+0,"No Error."'], 'raw': 'NEI+0.09080E-09', 'lop': 'LOP00,00,00,00,01,01,00,00,00,00', 'err_after': '+0,"No Error."'}
{'ch': 6, 'idn': 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401', 'err_before': ['+0,"No Error."'], 'raw': 'NFI+005.490E-12', 'lop': 'LOP00,00,00,00,00,01,00,00,00,00', 'err_after': '+0,"No Error."'}
{'ch': 7, 'idn': 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401', 'err_before': ['+0,"No Error."'], 'raw': 'NGI+0.00095E-09', 'lop': 'LOP00,00,00,00,00,00,01,00,00,00', 'err_after': '+0,"No Error."'}


## 看看两通道

In [5]:
import time
import pandas as pd

def two_channel_probe(b, force_ch, sense_ch, v_force=0.1, icomp=1e-3, delay_s=0.05):
    result = {
        "force_ch": force_ch,
        "sense_ch": sense_ch,
        "v_force": v_force,
        "raw": None,
        "current_A": None,
        "quality": None,
        "error": None,
    }

    try:
        b._write("FMT 1")
        time.sleep(delay_s)

        b._write("AV 10,1")
        time.sleep(delay_s)

        b._write("FL 0")
        time.sleep(delay_s)

        b.cn([force_ch, sense_ch])
        time.sleep(delay_s)

        # force_ch 加小偏压
        b.dv(force_ch, 0, v_force, icomp)
        time.sleep(delay_s)

        # sense_ch 保持 0V
        b.dv(sense_ch, 0, 0.0, icomp)
        time.sleep(delay_s)

        raw = b._query(f"TI {sense_ch},0", check_err=False)
        result["raw"] = raw

        val = b._parse_scalar_response(raw)
        result["current_A"] = val

        if abs(val) <= 1e-10:
            result["quality"] = "weak_or_off"
        else:
            result["quality"] = "responsive"

    except Exception as e:
        result["quality"] = "invalid"
        result["error"] = repr(e)

    finally:
        try:
            b.dz([force_ch, sense_ch])
        except Exception:
            pass

        try:
            b.cl([force_ch, sense_ch])
        except Exception:
            pass

    return result

In [ ]:
from fefetlab.instruments.visa_session import VisaSession, VisaConfig
from fefetlab.b1500 import B1500

cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=30000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

session = VisaSession(cfg)
session.open()
b = B1500(session)

print(session.query("*IDN?"))
print(b.errx())

In [7]:
channels = [4, 5, 6, 7]
rows = []

for force_ch in channels:
    for sense_ch in channels:
        if force_ch == sense_ch:
            continue

        print(f"\n=== force CH{force_ch} -> sense CH{sense_ch} ===")
        res = two_channel_probe(b, force_ch, sense_ch, v_force=0.1, icomp=1e-3, delay_s=0.05)

        print("raw      =", repr(res["raw"]))
        print("current  =", res["current_A"])
        print("quality  =", res["quality"])
        print("error    =", res["error"])

        rows.append(res)

df2 = pd.DataFrame(rows)

print("\n=== 汇总结果 ===")
print(df2)


=== force CH4 -> sense CH5 ===


NameError: name 'b' is not defined

In [8]:
df2_resp = df2[df2["quality"] == "responsive"].copy()
df2_weak = df2[df2["quality"] == "weak_or_off"].copy()
df2_invalid = df2[df2["quality"] == "invalid"].copy()

print("=== responsive ===")
print(df2_resp)

print("\n=== weak_or_off ===")
print(df2_weak)

print("\n=== invalid ===")
print(df2_invalid)

NameError: name 'df2' is not defined

In [ ]:
try:
    session.close()
except Exception as e:
    print("close session error:", e)